Inicio: Primera exploración de los datos de un solo mes (Enero del 2025).

- "Yellow taxis"

El archivo viene en formato parquet, muy utilizado en big data.
Para hacer una primera exploración se utilizará duckdb, ya que es mucho más rápido.

In [1]:
# Instalación librerias
!pip install -q duckdb

# Cargar librerías
import os, requests, duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Conectar a la carpeta del drive
from google.colab import drive

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/nyc-taxi"

Mounted at /content/drive


In [3]:
# Cargar los datos
con = duckdb.connect()
con.sql(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{BASE}/yellow_tripdata_*.parquet')")
  #El * del nombre sirve para que cuando haya más meses a la carpeta los lea todos.
con.sql(f"CREATE VIEW zones AS SELECT * FROM read_csv_auto('{BASE}/taxi_zone_lookup.csv')")


In [ ]:
# Cuantas filas hay
filas = con.sql("SELECT count(*) AS filas FROM trips")
print(filas)

# Columnas (cuantas hay y las etiquetas)
columnas = con.sql("DESCRIBE trips ")

print(columnas)

┌─────────┐
│  filas  │
│  int64  │
├─────────┤
│ 3475226 │
└─────────┘

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ VendorID              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_distance         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ RatecodeID            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_and_fwd_flag    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID          │ INTEGER     │ YES    

In [ ]:
# cuantos viajes por zona de origen
origen = con.sql("""SELECT PULocationID, count(*) AS viajes FROM trips
                    group by PULocationID
                    order by viajes desc""")
print(origen)

# cuantos viajes por zona de destinación
destino = con.sql("""SELECT DOLocationID, count(*) AS viajes FROM trips
                      group by DOLocationID
                      order by viajes desc""")
print(destino)

# Cuantos viajes hay por tipo de pago
pago = con.sql("""SELECT payment_type, count(*) AS viajes from trips
                  group by payment_type
                  order by viajes desc""")
print(pago)

# Cruce de pagos y origen
pagos_origen = con.sql("""SELECT PULocationID, payment_type, count(*) AS viajes FROM trips
                          group by PULocationID, payment_type
                          order by viajes desc""")
print(pagos_origen)

# Cruce de pagos y destinación
pagos_destino = con.sql("""SELECT DOLocationID, payment_type, count(*) AS viajes FROM trips
                            group by DOLocationID, payment_type
                            order by viajes desc""")
print(pagos_destino)

# Media de pago por zona de origen
media_origen = con.sql("""SELECT round(mean(Total_amount), 2) as pago_medio, PULocationID, count(*) AS viajes
                          from trips
                          group by PULocationID
                          order by pago_medio desc""")
print(media_origen)

# Media de pago por zona de destinación
media_destino = con.sql("""SELECT round(avg(Total_amount), 2) as pago_medio, DOLocationID, count(*) AS viajes
                            from trips
                            group by DOLocationID
                            order by pago_medio desc""")
print(media_destino)

# Descriptivos de distancia de los viajes
distancia = con.sql("""select count(*)  AS filas, count(trip_distance) AS no_nulos,
                        min(trip_distance) as minimo, max(trip_distance) as maximo,
                        round(avg(trip_distance), 2) as media, median(trip_distance) as mediana
                        from trips""")
print(distancia)

# Tiempo y distancia calculado por el aparato
fare = con.sql("""select count(*)  AS filas, count(fare_amount) AS no_nulos, min(fare_amount) as minimo,
                  max(fare_amount) as maximo, round(avg(fare_amount), 2) as media,
                  median(fare_amount) as mediana
                  from trips""")
print(fare)

#Pasajeros
pasajeros = con.sql("""select count(*)  AS filas, count(Passenger_count) AS no_nulos,
                        min(Passenger_count) as minimo, max(Passenger_count) as maximo,
                        round(avg(Passenger_count), 2) as media, median(Passenger_count) as mediana
                        from trips""")
print(pasajeros)

# Fechas
fechas = con.sql("""
SELECT strftime(tpep_pickup_datetime, '%Y-%m') AS mes, count(*) AS viajes
FROM trips GROUP BY mes ORDER BY mes
""")
print(fechas)

# Duración desplazamientos
duracion = con.sql("""
SELECT count(*) FILTER (WHERE tpep_dropoff_datetime <= tpep_pickup_datetime) AS duracion_cero_o_negativa,
       count(*) FILTER (WHERE date_diff('hour', tpep_pickup_datetime, tpep_dropoff_datetime) >= 5) AS mas_de_5_horas
FROM trips
""")
print(duracion)

# Duplicados
duplicados = con.sql("""
SELECT count(*) AS filas,
       count(DISTINCT (tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID, total_amount)) AS distintas
FROM trips
""")
print(duplicados)


┌──────────────┬────────┐
│ PULocationID │ viajes │
│    int32     │ int64  │
├──────────────┼────────┤
│          161 │ 169977 │
│          237 │ 163703 │
│          236 │ 155647 │
│          132 │ 146137 │
│          230 │ 125829 │
│          186 │ 119131 │
│          162 │ 117930 │
│          142 │ 110585 │
│          239 │  96614 │
│          163 │  95906 │
│           ·  │      · │
│           ·  │      · │
│           ·  │      · │
│          109 │      4 │
│           44 │      3 │
│          111 │      3 │
│          204 │      3 │
│            5 │      3 │
│          176 │      2 │
│          187 │      2 │
│           99 │      1 │
│          199 │      1 │
│           84 │      1 │
├──────────────┴────────┤
│  261 rows (20 shown)  │
└───────────────────────┘

┌──────────────┬────────┐
│ DOLocationID │ viajes │
│    int32     │ int64  │
├──────────────┼────────┤
│          236 │ 161376 │
│          237 │ 149970 │
│          161 │ 131258 │
│          230 │ 108177 │
│          

Hay un factor, donde hay más viajes de zona de origen y zona de destino pasa en las mismas zonas.
Hay muchos mas pago con tipo 1 (tarjeta de credito) tipo 0 (buscar q es) y tipo 2 (en efectivo).

Cuando cruzas ambas variables se encuntra el mismo factor. La gran mayoría de desplazamientos por las mismas zonas y pagando con trajeta de credito.

Media de distancia 5,86 millas. Hay valores 0 de distancia, mirar que pasa.

Media fare 17,08. Hay valores en negativo mirar que pasa.

Columna de pasajeros hay unas cuantas que tienen valor nulo.

In [ ]:
# COSAS A REVISAR PARA TENER EN CUENTA

# Distancia 0. Mira cómo son esos viajes comparados con el resto
distancia_0 = con.sql("""
SELECT count(*) AS viajes,
       round(100.0 * count(*) FILTER (WHERE PULocationID = DOLocationID) / count(*), 1) AS pct_misma_zona,
       round(avg(fare_amount), 2) AS fare_medio,
       round(avg(date_diff('minute', tpep_pickup_datetime, tpep_dropoff_datetime)), 1) AS minutos_medios
FROM trips
WHERE trip_distance = 0
""")
print(distancia_0)

# Distancia entre distintas zonas son 0
distancia_entre_zonas = con.sql("""
SELECT count(*) AS viajes,
       count(*) FILTER (WHERE PULocationID <> DOLocationID) AS distinta_zona,
       round(100.0 * count(*) FILTER (WHERE PULocationID <> DOLocationID) / count(*), 1) AS pct_distinta_zona
FROM trips
WHERE trip_distance = 0
""")
print(distancia_entre_zonas)

# Fare negativo. Suelen ser devoluciones, disputas o viajes anulados. Compruébalo con el tipo de pago
fare_negativo = con.sql("""
SELECT payment_type, count(*) AS viajes,
       round(avg(fare_amount), 2) AS fare_medio,
       round(avg(total_amount), 2) AS total_medio
FROM trips
WHERE fare_amount < 0
GROUP BY payment_type
ORDER BY viajes DESC
""")
print(fare_negativo)

con.sql("""
SELECT count(*) FILTER (WHERE fare_amount < 0)  AS fare_negativo,
       count(*) FILTER (WHERE fare_amount = 0)  AS fare_cero,
       count(*) FILTER (WHERE total_amount < 0) AS total_negativo,
       count(*) FILTER (WHERE fare_amount < 0 AND total_amount >= 0) AS fare_neg_total_pos
FROM trips
""").show()

# Pasajeros nulos
pasajeros_nulo = con.sql("""
SELECT payment_type,
       count(*) AS viajes,
       count(*) FILTER (WHERE passenger_count IS NULL) AS pasajeros_nulos,
       count(*) FILTER (WHERE RatecodeID IS NULL) AS ratecode_nulos,
       round(avg(total_amount), 2) AS total_medio
FROM trips
GROUP BY payment_type
ORDER BY payment_type
""")
print(pasajeros_nulo)

# Pasajeros nulos y tipo de pago 0
fare_negativo_distancia =con.sql("""
SELECT count(*) AS viajes,
       count(*) FILTER (WHERE fare_amount < 0) AS fare_negativo,
       round(avg(fare_amount), 2) AS fare_medio,
       round(avg(tip_amount), 2) AS propina_media,
       round(avg(trip_distance), 2) AS distancia_media
FROM trips
WHERE payment_type = 0
""")
print(fare_negativo_distancia)




┌────────┬────────────────┬────────────┬────────────────┐
│ viajes │ pct_misma_zona │ fare_medio │ minutos_medios │
│ int64  │     double     │   double   │     double     │
├────────┼────────────────┼────────────┼────────────────┤
│  90893 │           33.1 │      17.66 │           11.7 │
└────────┴────────────────┴────────────┴────────────────┘

┌────────┬───────────────┬───────────────────┐
│ viajes │ distinta_zona │ pct_distinta_zona │
│ int64  │     int64     │      double       │
├────────┼───────────────┼───────────────────┤
│  90893 │         60814 │              66.9 │
└────────┴───────────────┴───────────────────┘

┌──────────────┬────────┬────────────┬─────────────┐
│ payment_type │ viajes │ fare_medio │ total_medio │
│    int64     │ int64  │   double   │   double    │
├──────────────┼────────┼────────────┼─────────────┤
│            0 │  84822 │      -4.57 │        3.59 │
│            4 │  37405 │     -21.88 │      -27.75 │
│            2 │  13802 │     -22.66 │      -28.64

In [ ]:
# Mirar máximos de distancia
distancia_alta = con.sql("select trip_distance from trips order by trip_distance desc limit 125")
print(distancia_alta)

┌───────────────┐
│ trip_distance │
│    double     │
├───────────────┤
│     276423.57 │
│     276099.95 │
│     222167.49 │
│     206137.99 │
│     202771.63 │
│     189687.43 │
│     181139.99 │
│     168079.57 │
│     167452.94 │
│     164959.95 │
│          ·    │
│          ·    │
│          ·    │
│       1231.28 │
│         768.5 │
│        669.54 │
│         265.9 │
│        255.33 │
│        224.82 │
│        206.45 │
│         199.3 │
│        188.88 │
│        183.95 │
├───────────────┤
│   125 rows    │
│  (20 shown)   │
└───────────────┘



Son muy pocos los casos con distancia 0 (estos no se tendran en cuenta).

Tipo 0 (Flex Fare): 540.149 viajes (15,5 %). 15,7 % con fare negativo. Propina media 0,45 (poco fiable). Distancia media 20,31 millas, muy por encima del resto. Tira de la media global (5,86) hacia arriba; sin este tipo, la media sería ~3,2.
- Pendiente: mediana/p99 de distancia por tipo; ejemplos de fare negativo con total positivo.

- Fare negativo: 144.118 viajes (4,1 %). ~62.700 con total también negativo (ajustes, tipos 4/3/2) y 81.402 con total positivo (casi todos tipo 0).
- Fare = 0: 1.398 viajes (0,04 %).
- Total negativo: 63.037 viajes (1,8 %).
- Hipótesis: los totales negativos son reembolsos y podrían duplicar viajes al contar demanda. Por comprobar.

total_amount < 0 y excluirlos de los análisis de ingresos y de recuento de viajes.
En el tipo 0, usar total_amount y no fare_amount para medir el precio.
Marcar fare_amount = 0 como sospechoso, y mirar su tipo de pago.


Máximo de distancia poner 180 (este también tiene que ser el mismo para taxis verdes).




In [8]:
con.sql("""
SELECT count(*) AS viajes,
       count(*) FILTER (WHERE abs(total_amount - (
              fare_amount + extra + mta_tax + tip_amount + tolls_amount
              + improvement_surcharge + coalesce(congestion_surcharge, 0)
              + coalesce(Airport_fee, 0) + coalesce(cbd_congestion_fee, 0))) > 0.01
       ) AS no_cuadran
FROM trips
WHERE total_amount >= 0
""").df()

,viajes,no_cuadran
0,3412189,1082921


In [11]:
con.sql("""
WITH t AS (
  SELECT VendorID, congestion_surcharge, Airport_fee, cbd_congestion_fee,
         round(total_amount - (
           fare_amount + extra + mta_tax + tip_amount + tolls_amount
           + improvement_surcharge + coalesce(congestion_surcharge, 0)
           + coalesce(Airport_fee, 0) + coalesce(cbd_congestion_fee, 0)), 2) AS diferencia
  FROM trips
  WHERE total_amount >= 0
)
SELECT diferencia, VendorID, count(*) AS viajes,
       count(*) FILTER (WHERE congestion_surcharge IS NULL) AS cong_nulo,
       round(avg(coalesce(congestion_surcharge, 0)), 2) AS cong_media,
       round(avg(coalesce(cbd_congestion_fee, 0)), 2)   AS cbd_media
FROM t
WHERE diferencia IN (2.5, -3.25, -2.5, -1.75)
GROUP BY ALL
ORDER BY viajes DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,diferencia,VendorID,viajes,cong_nulo,cong_media,cbd_media
0,-3.25,1,391079,0,2.50,0.75
1,2.50,2,318343,314867,0.03,0.63
2,-2.50,1,176721,0,2.50,0.00
3,2.50,1,76270,76270,0.00,0.57
4,-1.75,1,15053,0,0.00,0.00
5,2.50,7,333,0,2.46,0.49


En el 32 % de los viajes amarillos, total_amount no coincide con la suma de sus componentes. Las diferencias son siempre importes exactos de tasas (2,50, 1,75, 0,75 y sus combinaciones), registradas de forma distinta según el proveedor. Decisión: usar total_amount y descartar el desglose.

**La carga de los datos tarda 3s.**

**El resto se ejecuta al instante.**

1
¿Dónde y cuándo hay más demanda?
Viajes por zona de recogida, hora y día de la semana
Excluir los ajustes (total_amount < 0) para no contar reembolsos como viajes

2
¿Qué zonas dan más ingreso por minuto?
(total_amount − tip_amount) / minutos del viaje
En el tipo 0, usar total_amount y no fare_amount. Quitar duraciones ≤ 0

3
¿Dónde acaban los taxis y dónde se necesitan?
Recogidas − destinos por zona y hora (flujo neto)
Una zona con más destinos que recogidas acumula taxis vacíos

4
¿Qué zonas dan mejor propina?
Propina / tarifa
Solo tarjeta, y sin el tipo 0. Mínimo de viajes por zona

5
¿Cómo se paga en cada zona?
% tarjeta / efectivo
Mostrar el tipo 0 aparte

6
¿Cuáles son las rutas origen-destino más importantes?
Viajes e ingreso por pareja de zonas
Mínimo de viajes por ruta

7
¿Cuánto pesan aeropuertos y viajes largos?
Viajes y tarifa por tipo de zona
Mira SELECT DISTINCT service_zone FROM zones